# Power analysis attack on ASCON

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import serial.tools.list_ports as port_list


# ---------------------------------------------------------------------------
# Project root discovery
# ---------------------------------------------------------------------------

def find_project_root(start: Path, markers=("fusesoc.conf", ".dojo_root")) -> Path:
    current = start
    while current != current.parent:
        if any((current / m).exists() for m in markers):
            return current
        current = current.parent

    raise RuntimeError(
        f"Could not find project root (looked for markers: {markers}). "
        "Please ensure you are inside the Side-Channel-Dojo repository."
    )


# In a script, __file__ exists; in a notebook, it does not.
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Notebook / interactive: use the current working directory instead
    SCRIPT_DIR = Path.cwd()

DOJO_ROOT = find_project_root(SCRIPT_DIR)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

ASCON_PY_DIR = DOJO_ROOT / "sw" / "ciphers" / "ASCON_init_python"
SCA_DIR      = DOJO_ROOT / "sw" / "sca_scripts"
HW_DIR       = DOJO_ROOT / "hw"

# Traces and plots for ASCON HW
TRACESET_DIR = DOJO_ROOT / "sw" / "traceset" / "ASCON" / "hw"
BASE_PLOT_DIR = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "hw" / "plot"

TRACESET_DIR.mkdir(parents=True, exist_ok=True)
BASE_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Make local modules importable without fragile ../ relative paths
sys.path.insert(0, str(ASCON_PY_DIR))
sys.path.insert(0, str(SCA_DIR))

# ---------------------------------------------------------------------------
# ASCON HW configuration
# ---------------------------------------------------------------------------

sub_layer_dict = {
    "hw":         "ASCON_HW",
    "lut_ascon":  "SBOX_ASCON",
    "lut_bilgin": "SBOX_BILGIN",
    "lut_allouzi": "SBOX_ALLOUZI",
    "lut_lu_4":   "SBOX_LU_4",
    "lut_lu_5":   "SBOX_LU_5",
    "lut_lu_6":   "SBOX_LU_6",
    "lut_lu_7":   "SBOX_LU_7",
}

# Select which ASCON S-box / sub-layer implementation to test
sub_layer_type = "hw"  # "hw", "lut_ascon", "lut_bilgin", ...

# CW305 project (.cwp) and bitstream paths, rooted at DOJO_ROOT
project_file_path = (
    DOJO_ROOT
    / "build"
    / "sca_test"
    / f"sca_test_CW305_ascon_init_{sub_layer_dict[sub_layer_type]}_300000_1.cwp"
)

bitstream_path = (
    HW_DIR
    / "fpga"
    / "bitstream"
    / f"cw305_top_ascon_init_{sub_layer_dict[sub_layer_type]}.bit"
)

project_file = str(project_file_path)
bitstream = str(bitstream_path)

# Directory for plots for this sub-layer
PLOT_DIR = BASE_PLOT_DIR / sub_layer_dict[sub_layer_type]
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Helper for pretty yes/no printing
# ---------------------------------------------------------------------------

def _yn(flag: bool) -> str:
    return "yes" if flag else "no"


# ---------------------------------------------------------------------------
# Serial ports & configuration printout
# ---------------------------------------------------------------------------

ports = list(port_list.comports())
print("\n================= AVAILABLE SERIAL PORTS =================")
for p in ports:
    print(f"  {p}")
print("=========================================================\n")

print("\n================= ASCON HW CONFIGURATION =================")
print(f"DOJO_ROOT           : {DOJO_ROOT}")
print()
print("Target")
print(f"  Sub-layer type    : {sub_layer_type}")
print(f"  Sub-layer label   : {sub_layer_dict[sub_layer_type]}")
print()
print("Paths")
print(f"  Bitstream         : {bitstream}")
print(f"  CW project (.cwp) : {project_file}")
print(f"  Traceset dir      : {TRACESET_DIR}")
print(f"  Plot dir          : {PLOT_DIR}")
print("==========================================================\n")

## Picoscope and CW305 initialization

In [ ]:
from pico_api import PS5000aWrapper
from CW305_api import CW305Wrapper

try:
    # Initialize picoscope
    ps = PS5000aWrapper()
    ps.get_unitInfo()
    ps.scope_setup()
    # Initialize CW305
    cw305 = CW305Wrapper(ps, bitstream)

except ModuleNotFoundError as e:
    print(e)

## Online phase : ASCON power traces capture

In [ ]:
from tqdm.notebook import tnrange
import chipwhisperer as cw
from operations_init import ascon_init as ascon

# Number of traces to capture
N = 60000

project_file = "../../build/sca_test/sca_test_CW305_ascon_init_" + sub_layer_dict[sub_layer_type] + "_" + str(N) + ".cwp"
project = cw.create_project(project_file, overwrite=True)

key = 0x000102030405060708090A0B0C0D0E0F
key_str = f"{key:032x}"
key_list = [int(key_str[i:i+2], 16) for i in range(0, len(key_str), 2)]

ktp = cw.ktp.Basic()
kk, nonce = ktp.next()

# Each element of the key is converted to a 2-digit hex string 
print("Key: ", [ hex(subkey) for subkey in key_list])

# Write the key to the CW305
cw305.set_key(key_list)
# Dummy capture call due to bug of using AC coupling
cw305.capture_trace_1_round(project, nonce, dummy=True)
    
for i in tnrange(N, desc='Capturing traces'):
    response = cw305.capture_trace_1_round(project, nonce)

    # Sanity check with expected ciphertext
    state = ascon(key.to_bytes(16,'big'), nonce, "Ascon-128", sub_layer_type)
    assert (list(state) == list(response)), "Incorrect encryption result!\nGot {}\nExp {}\n".format(list(response), list(state))
    kk, nonce = ktp.next()

project.save()
project.close()
# Disconnect CW305 and picoscope
cw305.dis()
ps.dis()